In [ ]:
import tensorflow as tf
import datetime, os
import logging

In [ ]:
!apt-get install x11-utils > /dev/null 2>&1
!pip install pyglet > /dev/null 2>&1
!apt-get install -y xvfb python-opengl > /dev/null 2>&1
!pip install gym pyvirtualdisplay > /dev/null 2>&1

In [ ]:
!pip install gym[atari] gym[accept-rom-licence] atari_py

In [ ]:
import ray
import ray.rllib.agents.dqn as dqn
from ray.rllib.agents.dqn.dqn import DEFAULT_CONFIG

# Use for Vanilla DQN
duelddqn_config = dqn.DEFAULT_CONFIG.copy()


 # Works for both torch and tf.
duelddqn_config['framework'] = 'tf2'
duelddqn_config['double_q'] = True
duelddqn_config['dueling']: True
duelddqn_config['num_atoms'] = 1
duelddqn_config['noisy'] = False
duelddqn_config['prioritized_replay'] = False
duelddqn_config['n_step'] = 3
duelddqn_config['target_network_update_freq'] = 32000
duelddqn_config['lr']: .0000625
duelddqn_config['adam_epsilon']: .00015
duelddqn_config['hiddens'] [512]
duelddqn_config['learning_starts'] = 80000
duelddqn_config['buffer_size'] = 100000
duelddqn_config['rollout_fragment_length'] = 4
duelddqn_config['train_batch_size'] = 32
duelddqn_config['timesteps_per_iteration'] = 10000
duelddqn_config['num_gpus'] = 1
duelddqn_config['exploration_config'] = {"epsilon_timesteps": 200000, "final_epsilon": 0.01}
duelddqn_config['model'] = {"fcnet_hiddens": [512], "fcnet_activation": 'relu', 'grayscale':True, "zero_mean": False, "dim": 84, "framestack":False}      
duelddqn_config['v_min'] = -10
duelddqn_config['v_max'] = 10

from ray import tune

if ray.is_initialized:
    ray.shutdown()

# To disable ray workers from logging the output.

ray.init(include_dashboard=False, ignore_reinit_error=True, log_to_driver=False,num_cpus=4) 


metric = 'episode_reward_mean'
mode = 'max'

import logging

analysis_duel_ddqn = tune.run(dqn.DQNTrainer,
                    name="SpaceInvadersDuelDDQN",
                    num_samples=1,
                    config=duelddqn_config,
                    metric=metric,
                    mode=mode,
                    local_dir="./drive/MyDrive",
                    stop={"episode_reward_mean": 500, "training_iteration": 80},
                    checkpoint_at_end=True)


with open('analysis_duel_ddqn.pickle', 'wb') as f:
    pickle.dump([analysis_duel_ddqn], f)



In [ ]:
best_trial_duelddqn = analysis_duel_ddqn.get_best_trial(metric="episode_reward_mean", mode="max", scope="all") 

checkpoint_duelddqn = analysis_duel_ddqn.get_best_checkpoint(best_trial_duelddqn,"training_iteration","max")
trainer_duelddqn = dqn.DQNTrainer(config=duelddqn_config)
trainer_duelddqn.restore(best_trial_duelddqn)